# **Mineral Prospectivity Project**
## 01 data ingestion 

goals:\
-load Idaho State Survey and USGS data\
-identify and filter to data relevant to gold mining and prospectivity\
-export gold data subset for further analysis\
\
-load USGS National Geochemical Database (NGDB) data for Idaho\
-interrogate and convert to spaital analysis-ready geodataframes


### Part 1. import packages and identify local directory
a. import packages needed for data loading and analysis

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import glob
import os

data_path = "/Users/adbyerly/prospectivity_model/data/"

b. metadata dataframe for sources\
-create dataframe\
-add source URLs\
-add notes about cleaning/processing & update with further work

In [ ]:
#metadata

sources = pd.DataFrame({
    'source' : [
        'Idaho Geologic Survey',
        'USGS',
        'USGS',
        'USGS',
    ],
    
    'table_name' : [
        'mines',
        'ngdb',
        'mrds',
        'magnetic'
    ],

    'source_url' : [
        'https://www.idahogeology.org/product/dd-1',
        'https://www.usgs.gov/centers/gggsc/science/national-geochemical-database',
        'https://mrdata.usgs.gov/mrds/',
        'https://mrdata.usgs.gov/magnetic/'
    ],

    'donwload_date' : [
        '2026-05-01',
        '2026-05-01',
        '2026-05-06',
        '2026-05-08'
    ],

    'original_datum' : [
        'WGS84',
        'Mixed(WGS84, NAD83, NAD27, NaN',
        '',
        'NAD27'
    ],

    'notes' : [
        None,
        'projected unknonwn datum values to NAD27 per metadata',
        None,
        None
    ]})

sources.head()

### Part 2. load raw data and process

a. load raw data from Idaho Geologic Survey mines database\
-load data\
-inspect format and identify data relevant to gold\
-filter to gold data\
-export layer file for future analysis

In [ ]:
#shapefile import
mines_gdf = gpd.read_file(data_path + "raw/MinesPubDD-1_2023/MinesPubDD-1_Shapefile_2023/Mines_DD-1_2023.shp")
mines_gdf.head()

In [ ]:
mines_gdf.shape

In [ ]:
mines_gdf.columns

In [ ]:
mines_gdf[mines_gdf["CommodityL"].str.contains("gold", case=False, na=False)]["CommodityL"].value_counts()

In [ ]:
#create a subset of this dataframe that only includes data relevant to gold
gold_gdf = mines_gdf[mines_gdf["CommodityL"].str.contains("gold", case=False, na=False)]
gold_gdf.head()

In [ ]:
# save gold layer
gold_gdf.to_file(data_path + "processed/idaho_gold_mines.geojson", driver="GeoJSON")

b. load raw geochemical data from the "USGS National Geochemical Database (NGDB)"\
-data filtered to the state of Idaho before bulk download from USGS website\
-load data\
-interrogate data and datums\
-standaradize CRS

In [ ]:
# csv - load all geochemical files from the downloaded folder into a dictionary
files = glob.glob(data_path + "raw/ngdbrock-fUS16/*.txt")
ngdb_dfs={}

for file in files:
    name = os.path.basename(file).replace(".txt", "")
    df = pd.read_csv(file, low_memory=False)
    ngdb_dfs[name] = df

ngdb_dfs.keys()

In [ ]:
# rename dictionary items to more useful names based on metadata here: https://mrdata.usgs.gov/metadata/ngdbrock.html
renaming = {"tblRockGeoData" : "Rock_Data", #rock sample data
            "xtbOtherChem" : "Other_chem", #atomic absorption spectrometry, colorimetry, ion selective electrode (except Cl & F), fluorometry, or fire assay
            "xtbEsChem" : "ES", #electron spectroscopy
            "xtbUnknownChem" : "Unknown_chem", #analyses derived from undocumented methods
            "xtbIcpmsChem" : "ICPMS", #inductively coupled plasma mass spectrometry
            "xtbIcpaesChem" : "ICPAES", #inductively coupled plasma-atomic emission spectrometry
            "xtbMajorChem" : "Major_element", #major element oxide analyses plus forms-of-carbon, forms-of-sulfur, chlorine, and fluorine data.
            "xtbXrfChem" : "XRF", #x-ray fluorescence
            "xtbNaaChem" : "NAA" #instrumental neutron activation analysis or delayed neutron activation analysis
           }

ngdb = {renaming.get(k, k): v for k, v in ngdb_dfs.items()} 
ngdb.keys()

In [ ]:
# examine dataframes; explanation of tables and features is here: https://mrdata.usgs.gov/ngdb/rock/about.php
# primary key for samples is "LAB_ID", georeferenced in 'Rock_Data' table and used as a foreign key in all other tables

for name, df in ngdb.items():
    print(name)
    print(df.head())
    print(df.columns)
    print(df.shape)
    print("-" * 25)

In [ ]:
# process NGDB data

# 'Rock_Data'
# first we need to know the datum for lat and long values; 
# from the metadata, legacy data are North American Datum of 1927 on the Clarke 1866 ellipsoid (https://mrdata.usgs.gov/metadata/ngdbrock.faq.html)

print(ngdb['Rock_Data']['datum'].unique())
print(ngdb['Rock_Data']['datum'].value_counts(dropna=False))

# --- UNCERTAINTY  --- 
# -------------------- it is unclear if ALL of the data are NAD27; there are a large amount of NaN values
# -------------------- assume that these are all NAD27 for regional analysis 
# -------------------- investigate further and test alternate scenarios for detailed work

# investigate vintage of samples with undefined datum
rock = ngdb['Rock_Data']

dates = pd.to_datetime(
    pd.to_numeric(
        rock.loc[rock['datum'].isna(), 'date_sub'], errors = 'coerce'),
    format = '%Y%m%d', errors = 'coerce')

dates.agg(['min', 'max'])


In [ ]:
# un-datumed samples range in submission date from 1964 to 2001
# assume these are NAD 1927 per metadata but note uncertainty highlighted above

rock['datum'] = rock['datum'].fillna('NAD27')
rock.head()


In [ ]:
# create geopandas dataframes for sample table ('Rock_Data'), split by datum

epsg = {
    'NAD27' : 4267,
    'WGS84' : 4326,
    'NAD83' : 4269
}

# dicitonary of 'Rock_Data' split into geodataframe by datum (3 datums)
# these dataframes won't be used for analysis, but serve as a record of and are ready for subsequent reprojection

rock_gdfs = {}

for d, df in rock.groupby('datum'):
    rock_gdfs[d] = gpd.GeoDataFrame(
        df.copy(),
        geometry = gpd.points_from_xy(df['longitude'], df['latitude']),
        crs = f"EPSG:{epsg[d]}")

In [ ]:
# project geodataframes in dictionary to common CRS (Idaho Transverse Mercator (ITM))
target_crs = 'EPSG:8826'

rock_gdfs_ITM = {key: gdf.to_crs(target_crs) for key, gdf in rock_gdfs.items()}

# concatenate reprojected geodataframes into one dataframe
rock_ITM = pd.concat(rock_gdfs_ITM.values(), ignore_index=True)

# preserve origional datum and add column to reference new datum
rock_ITM.rename(columns={'datum':'original_datum'}, inplace = True)
rock_ITM['projected_datum'] = 'ITM'

print(rock_ITM.head())
print(rock_ITM.shape)

In [ ]:
# standardized dictionary with base table converted to one CRS ready for analysis
ngdb_itm = {k: v.copy() for k, v in ngdb.items()}
ngdb_itm['Rock_Data'] = rock_ITM.copy()
ngdb_itm.keys()

In [ ]:
ngdb_itm['Rock_Data'].head()

c. load data from the USGS Mineral Resource Data System (MRDS)
-data filtered to the state of Idaho before bulk download from USGS website\
-load data\
-interrogate data and datums\
-convert to GeoDataFrame

In [ ]:
# CSV import
mrds = pd.read_csv(data_path + 'raw/MRDS/mrds-fUS16.txt', low_memory=False)

In [163]:
print(mrds.head())
print(mrds.columns)
print(mrds.shape)

     dep_id                                                url  mrds_id  \
0  10008100  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000011   
1  10008372  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000470   
2  10008373  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000472   
3  10008375  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000474   
4  10008376  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000477   

  mas_id                site_name  latitude  longitude  region        country  \
0    NaN            Buckskin Mine  43.98321 -115.88429     NaN  United States   
1    NaN             Little Falls  44.08321 -115.75096     NaN  United States   
2    NaN  Boundary County Deposit  48.88319 -116.83441     NaN  United States   
3    NaN            Boulder Creek  44.05179 -114.56232     NaN  United States   
4    NaN                 Sawtooth  44.03549 -114.97673     NaN  United States   

   state  ...                                                r

In [164]:
mrds['dep_type'].unique()

<StringArray>
[                                                nan,
                                            'Placer',
                          'HYDROTHERMAL BEDDED VEIN',
                                              'Vein',
                                       'Replacement',
                                       'Sedimentary',
                                       'Metamorphic',
                                           'Unknown',
                                     'Stream Placer',
                                             'Float',
 ...
          'SYNGENETIC SEDIMENT-HOSTED (CARBON-RICH)',
                                     'STREAM PLACER',
                             'VEINS AND SHEAR ZONES',
                                   'VEIN/SHEAR ZONE',
                 'VEIN, SHEAR ZONE CHIMNEY OR STOCK',
                                 'VEIN, REPLACEMENT',
                           'STRATIFORM DISSEMINATED',
 'HYDROTHERMAL VEINS, STOCKWORK, CHIMNEYS, BRECCIAS',
         

In [ ]:
# convert to GeoDataFrame
mrds_gdf = gpd.GeoDataFrame(
    mrds.copy(), 
    geometry = gpd.points_from_xy (mrds['longitude'], mrds['latitude']),
    crs = 'EPSG:4326'
)

mrds_gdf.head()